# 01 - EDA

This notebook explores the WESAD wrist signals before any preprocessing or feature extraction.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.loader import describe_subject, list_subjects, load_subject

In [ ]:
subject_ids = list_subjects()
print(f"Subjects found: {len(subject_ids)}")
subject_ids

## Inspect one subject

In [ ]:
subject_id = "S2"
subject_summary = describe_subject(subject_id)

display(pd.DataFrame([subject_summary]))
display(pd.DataFrame(subject_summary["wrist_signals"]).T)

## Signal summary across all subjects

In [ ]:
rows = []

for subject_id in subject_ids:
    subject = load_subject(subject_id)
    for signal_name, values in subject.wrist_signals.items():
        rows.append(
            {
                "subject_id": subject_id,
                "signal": signal_name,
                "samples": values.shape[0],
                "channels": values.shape[1] if values.ndim > 1 else 1,
            }
        )

signal_summary = pd.DataFrame(rows)
display(signal_summary.head())
display(signal_summary.groupby("signal")[["samples", "channels"]].agg(["min", "max", "mean"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
signal_summary.boxplot(column="samples", by="signal", ax=ax)
ax.set_title("Sample count by wrist signal")
ax.set_ylabel("Number of samples")
plt.suptitle("")
plt.tight_layout()

## Label distribution

In [ ]:
label_name_map = {
    1: "baseline",
    2: "stress",
    3: "amusement",
    4: "meditation",
}

label_rows = []
for subject_id in subject_ids:
    labels = load_subject(subject_id).labels.astype(int).ravel()
    counts = pd.Series(labels).value_counts().sort_index()
    for label_value, count in counts.items():
        label_rows.append(
            {
                "subject_id": subject_id,
                "label": int(label_value),
                "label_name": label_name_map.get(int(label_value), f"label_{int(label_value)}"),
                "count": int(count),
            }
        )

label_summary = pd.DataFrame(label_rows)
display(label_summary.head())
display(label_summary.groupby(["label", "label_name"], as_index=False)["count"].sum())

In [ ]:
target_label_summary = (
    label_summary[label_summary["label"].isin([1, 2, 3])]
    .pivot(index="subject_id", columns="label_name", values="count")
    .fillna(0)
)

display(target_label_summary)

ax = target_label_summary.plot(kind="bar", stacked=True, figsize=(12, 5))
ax.set_title("Label distribution per subject")
ax.set_xlabel("Subject")
ax.set_ylabel("Number of label samples")
plt.tight_layout()

## Plot a short raw segment

In [ ]:
subject_id = "S2"
subject = load_subject(subject_id)

plot_order = ["ACC", "BVP", "EDA", "TEMP"]
sample_limits = {
    "ACC": 2000,
    "BVP": 4000,
    "EDA": 1000,
    "TEMP": 1000,
}

fig, axes = plt.subplots(len(plot_order), 1, figsize=(14, 10))

for ax, signal_name in zip(axes, plot_order):
    values = subject.wrist_signals[signal_name]
    limit = min(sample_limits[signal_name], len(values))
    segment = values[:limit]

    if segment.ndim == 2 and segment.shape[1] > 1:
        for channel_index in range(segment.shape[1]):
            ax.plot(segment[:, channel_index], label=f"channel_{channel_index}")
        ax.legend(loc="upper right", ncol=3, fontsize=8)
    else:
        ax.plot(segment.reshape(-1))

    ax.set_title(f"{subject_id} - {signal_name} - first {limit} samples")
    ax.grid(alpha=0.3)

plt.tight_layout()

## Notes

- Record what labels are actually present in the dataset.
- Check whether the class balance is acceptable for the chosen task.
- Use these observations to design preprocessing and windowing in the next step.